# Tutorial 1: nanoGPT - word-Level Text Generation

Welcome to the nanoGPT tutorial! In this notebook, we will train a simple word-level language model to generate text.

**Goal:** Understand how to train a small transformer model for text generation on a CPU.

**Key Concepts:**
- Word-level tokenization
- Preparing data for language modeling
- Building a GPT-style transformer from scratch (simplified)
- A basic PyTorch training loop
- Generating text with the trained model

## 1. Setup and Imports

First, let's import the necessary libraries. We'll need `torch` for building and training our model.

In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import GPT2Config, GPT2LMHeadModel
import math
import random
import numpy as np

# For reproducibility
random.seed(1)
np.random.seed(1)
torch.manual_seed(1)

## 2. Configuration

We'll define some hyperparameters for our model and training process. Since we're running on CPU and want this to be quick for a tutorial, these values will be small.

In [12]:
# Hyperparameters
batch_size = 32       # How many independent sequences will we process in parallel 
block_size = 10      # What is the maximum context length for predictions?
max_iters = 5000      # Total training iterations
eval_interval = 500   # How often to evaluate on validation set
learning_rate = 5e-4  # Learning rate for the optimizer
device = 'cpu'        # Explicitly set to CPU
eval_iters = 100      # Number of iterations for evaluation
n_embd = 128          # Embedding dimension (reduced for CPU)
n_head = 4            # Number of attention heads (reduced for CPU)
n_layer = 4           # Number of transformer blocks (reduced for CPU)
dropout = 0.0          # Dropout rate>0 to prevent overfitting

## 3. Data Preparation

We'll use a small text file as our dataset. For this tutorial, let's use a snippet of Shakespeare's writings.

### 3.1 Load Data

In [13]:
# We'll use a small snippet of text for this tutorial.
# You can replace this with the path to a larger .txt file if you wish.
text = """To stale 't a little more.

First Citizen:
Well, I'll hear it, sir: yet you must not think to
fob off our disgrace with a tale: but, an 't please
you, deliver.

MENENIUS:
There was a time when all the body's members
Rebell'd against the belly, thus accused it:
That only like a gulf it did remain
I' the midst o' the body, idle and unactive,
Still cupboarding the viand, never bearing
Like labour with the rest, where the other instruments
Did see and hear, devise, instruct, walk, feel,
And, mutually participate, did minister
Unto the appetite and affection common
Of the whole body. The belly answer'd--

First Citizen:
Well, sir, what answer made the belly?

MENENIUS:
Sir, I shall tell you. With a kind of smile,
Which ne'er came from the lungs, but even thus--
For, look you, I may make the belly smile
As well as speak--it tauntingly replied
To the discontented members, the mutinous parts
That envied his receipt; even so most fitly
As you malign our senators for that
They are not such as you."""

### 3.2 Word-level Tokenization

Since this is a word-level model, our vocabulary will consist of all unique words present in the text. We'll create mappings from words to integers (encode) and integers to words (decode).

In [14]:
# Get all unique words in the text
import re

# Simple word tokenization - split on whitespace and punctuation
words = re.findall(r'\b\w+\b|[^\w\s]', text.lower())
unique_words = sorted(list(set(words)))
vocab_size = len(unique_words)

print("Sample words:", unique_words[:20])
print(f"Vocabulary size: {vocab_size}")

# Create a mapping from words to integers and vice-versa
stoi = {word: i for i, word in enumerate(unique_words)}
itos = {i: word for i, word in enumerate(unique_words)}

def encode(s):
    """Encoder: take a string, output a list of integers"""
    words = re.findall(r'\b\w+\b|[^\w\s]', s.lower())
    return [stoi[word] for word in words if word in stoi]

def decode(l):
    """Decoder: take a list of integers, output a string"""
    words = [itos[i] for i in l]
    # Simple reconstruction - join with spaces, handle punctuation
    result = ""
    for i, word in enumerate(words):
        if word in ".,!?;:":
            result += word
        elif i == 0:
            result += word
        else:
            result += " " + word
    return result

# Test encoding and decoding
test_text = "an answer"
encoded = encode(test_text)
decoded = decode(encoded)
print(f"Original: {test_text}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")

Sample words: ["'", ',', '-', '.', ':', ';', '?', 'a', 'accused', 'affection', 'against', 'all', 'an', 'and', 'answer', 'appetite', 'are', 'as', 'bearing', 'belly']
Vocabulary size: 123
Original: an answer
Encoded: [12, 14]
Decoded: an answer


### 3.3 Create Training and Validation Splits

We'll split our dataset into a training set and a validation set. The model learns from the training set, and we use the validation set to check how well it's generalizing.

In [15]:
# Encode the entire text dataset and store it into a torch.Tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}, Data type: {data.dtype}")
print(f"Sample encoded text: {data[:20].tolist()}")
print(f"Sample decoded text: {decode(data[:20].tolist())}")

# Split up the data into train and validation sets
n = int(0.5*len(data)) # first 80% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(f"Training data length: {len(train_data)} words")
print(f"Validation data length: {len(val_data)} words")

Data shape: torch.Size([238]), Data type: torch.int64
Sample encoded text: [108, 94, 0, 97, 7, 52, 64, 3, 36, 23, 4, 114, 1, 44, 0, 53, 42, 48, 1, 90]
Sample decoded text: to stale ' t a little more. first citizen: well, i ' ll hear it, sir
Training data length: 119 words
Validation data length: 119 words


### 3.4 Data Loader

We need a way to feed data to our model in batches. The `get_batch` function will randomly sample `batch_size` chunks of `block_size` length from the data.

In [16]:
#  Data loading function 
def get_batch(split):
    # Generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Example of a batch
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print('targets:')
print(yb.shape)

print('----')
print("Word-level training examples:")
for b in range(min(2, batch_size)):  # Show 2 examples
    for t in range(min(8, block_size)):  # Show first 8 positions
        context_words = decode(xb[b, :t+1].tolist())
        target_word = itos[yb[b,t].item()]
        print(f"Context: '{context_words}' -> Target: '{target_word}'")
        if t == 7:  # Only show first 8 examples
            break
    print()
    if b == 1:  # Only show for first 2 batch items
        break

inputs:
torch.Size([32, 10])
targets:
torch.Size([32, 10])
----
Word-level training examples:
Context: 'instruments' -> Target: 'did'
Context: 'instruments did' -> Target: 'see'
Context: 'instruments did see' -> Target: 'and'
Context: 'instruments did see and' -> Target: 'hear'
Context: 'instruments did see and hear' -> Target: ','
Context: 'instruments did see and hear,' -> Target: 'devise'
Context: 'instruments did see and hear, devise' -> Target: ','
Context: 'instruments did see and hear, devise,' -> Target: 'instruct'

Context: 'only' -> Target: 'like'
Context: 'only like' -> Target: 'a'
Context: 'only like a' -> Target: 'gulf'
Context: 'only like a gulf' -> Target: 'it'
Context: 'only like a gulf it' -> Target: 'did'
Context: 'only like a gulf it did' -> Target: 'remain'
Context: 'only like a gulf it did remain' -> Target: 'i'
Context: 'only like a gulf it did remain i' -> Target: '''



## 4. Model Definition (Simplified nanoGPT Style)

Now we'll build a simplified version of the GPT model.

In [17]:
# Define the GPT-2 model configuration using transformers library
config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,  # Max sequence length
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout,
    bos_token_id=None,  # No beginning of sentence token for char model
    eos_token_id=None   # No end of sentence token for char model
)

# Instantiate the model
model = GPT2LMHeadModel(config)
model.to(device)

# Print the number of parameters in the model
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters")

0.81 M parameters


## 5. Training the Model

### 5.1 Optimizer
We'll use the AdamW optimizer, a common choice for training transformer models.

In [18]:
# Create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

### 5.2 Loss Estimation
We need a function to estimate the loss on the training and validation sets without calculating gradients, which is useful for monitoring training progress.

In [19]:
@torch.no_grad() # Decorator to disable gradient calculation
def estimate_loss():
    out = {}
    model.eval() # Set model to evaluation mode
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            outputs = model(X, labels=Y)
            loss = outputs.loss
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # Set model back to training mode
    return out

### 5.3 Training Loop
This is the main loop where the model learns. For each iteration, we:
1. Get a batch of data.
2. Perform a forward pass (get model predictions).
3. Calculate the loss (how wrong the predictions are).
4. Perform a backward pass (calculate gradients).
5. Update the model's parameters using the optimizer.

In [20]:
print(f"Training on {device}...")

for iter_num in range(max_iters):

    # Every once in a while, evaluate the loss on train and val sets
    if iter_num % eval_interval == 0 or iter_num == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Evaluate the loss
    outputs = model(xb, labels=yb)
    loss = outputs.loss
    
    # Backward pass and optimization step
    optimizer.zero_grad(set_to_none=True) # Zero out gradients from previous step
    loss.backward() # Compute gradients for the current batch
    optimizer.step() # Update model parameters

print("Training finished!")

Training on cpu...


step 0: train loss 4.8432, val loss 4.8423
step 500: train loss 0.1071, val loss 6.8329
step 1000: train loss 0.0881, val loss 7.4115
step 1500: train loss 0.0837, val loss 7.8524
step 2000: train loss 0.0821, val loss 8.1622
step 2500: train loss 0.0795, val loss 8.4942
step 3000: train loss 0.0800, val loss 8.7935
step 3500: train loss 0.0789, val loss 8.9024
step 4000: train loss 0.0810, val loss 8.9835
step 4500: train loss 0.0770, val loss 9.2195
step 4999: train loss 0.0838, val loss 9.3415
Training finished!


## 6. Generating Text

Now that our model is trained, let's use it to generate some text. We'll start with a context (e.g., a newline word) and ask the model to predict the next words sequentially.

In [26]:
# Generation function - create a simple generation loop
print("Generating text...")
model.eval()

# Start with the first few words from our training data
start_words = block_size
text_test=" ".join(decode(train_data[:start_words].tolist()).split()[:start_words])
text_test="sir: yet you must not think to fob off our disgrace with a tale"
context = torch.tensor(encode(text_test), 
                      dtype=torch.long, device=device).unsqueeze(0)

print(f"Starting context: '{decode(context[0].tolist())}'")

# Simple generation loop
generated = context
with torch.no_grad():
    for _ in range(50):  # Generate 50 more words
        # Get predictions for the current context
        # Crop to last block_size tokens if context gets too long
        context_cropped = generated[:, -block_size:] if generated.size(1) > block_size else generated
        
        outputs = model(context_cropped)
        logits = outputs.logits
        
        # Get the logits for the last token
        logits = logits[:, -1, :]  # (batch_size, vocab_size)
        
        # Apply temperature for sampling
        temperature = 1
        logits = logits / temperature
        
        # Sample from the distribution
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        # Append to generated sequence
        generated = torch.cat([generated, next_token], dim=1)

# Decode the generated text
generated_text = decode(generated[0].tolist())
print("--- Generated Text ---")
print(generated_text)
print("--- Original Text ---")
print('''First Citizen:
Well, I'll hear it, sir: yet you must not think to
fob off our disgrace with a tale: but, an 't please
you, deliver.

MENENIUS:
There was a time when all the body's members
Rebell'd against the belly, thus accused it:
That only like a gulf it did remain''')
print("----------------------")

Generating text...
Starting context: 'sir: yet you must not think to fob off our disgrace with a tale'
--- Generated Text ---
sir: yet you must not think to fob off our disgrace with a tale but an t you deliver menenius there a more first: was time all body s rebell d the, ' against belly thus it that like gulf did i the o the,:, the, the, ' and, ' please, ' hear,,
--- Original Text ---
First Citizen:
Well, I'll hear it, sir: yet you must not think to
fob off our disgrace with a tale: but, an 't please
you, deliver.

MENENIUS:
There was a time when all the body's members
Rebell'd against the belly, thus accused it:
That only like a gulf it did remain
----------------------
